In [1]:
import ee
import geemap
import json
import os
#from geemap import geojson_to_ee
from ipyleaflet import GeoJSON
import segment_testing_my_gee_functions as mgf

In [3]:
ee.Authenticate()
#Initializing the ee project

ee.Initialize(project = 'ee-jdawsey')

In [4]:
lek_shp_path = 'E:/!!Research/!!!Data/lek_landcover_class/lek_landcover_py/lek_area_fishnet_nm/lek_nm_merge.shp'
training_points_path = 'E:/!!Research/!!!Data/lek_landcover_class/lek_landcover_py/training_points/nm_training_points.shp'
training_data = geemap.shp_to_ee(training_points_path)

In [8]:
# create a feature collection from the feature
feature_collection = geemap.shp_to_ee(lek_shp_path)
# find the features geometry so can be used for calculations 
feature_geom = feature_collection.geometry()

naip = mgf.naip_func(feature_collection)

savi = mgf.naip_savi(naip)

def gauss_three_func(given_image):
    ### gaussian smoothing
    gaussianKernel = ee.Kernel.gaussian(
      radius = 3, # was originally 3, also tested 10
      units = 'pixels'
    )
    
    gaussian_smooth = given_image.convolve(gaussianKernel)
    return gaussian_smooth

gauss = gauss_three_func(savi)
vis_param_smooth = {'bands' : ['R', 'G', 'B'], 
               'min' : 0, 
               'max' : 256,
               'gamma' : 1}

# creating a grayscale image for the glcm
grayscale = naip.expression(
      '(0.3 * R) + (0.59 * G) + (0.11 * B)', {
      'R': naip.select(['R']),
      'G': naip.select(['G']),
      'B': naip.select(['B'])
})

# if wanting to use a grayscale image
int_gray = grayscale.int()
glcmTexture = int_gray.glcmTexture(5)

# selecting just those necessary bands
imp_glcm_bands = glcmTexture.select(['constant_savg', 'constant_contrast'])

# adding the rng bands back
glcm_segment = imp_glcm_bands.addBands(gauss, ['R', 'G', 'B', 'N', 'savi']) #may need to include 'savi' and 'endvi'

# standardizing the imagery
standardized = mgf.stdrd_func(glcm_segment, feature_collection)

# Sample training points
sample = standardized.sampleRegions(
    collection=training_data,
    properties=['Class'],  # Use your class label column
    scale=1
)
training_sample = sample.filter('random <= 0.8')
validation_sample = sample.filter('random > 0.8')

# Train a 100-tree random forest classifier from the training sample.
trained_classifier = ee.Classifier.smileRandomForest(100).train(
    features=training_sample,
    classProperty='class',
    inputProperties=standardized.bandNames(),
)
"""
# Get information about the trained classifier.
display('Results of trained classifier', trained_classifier.explain())

# Get a confusion matrix and overall accuracy for the training sample.
train_accuracy = trained_classifier.confusionMatrix()
display('Training error matrix', train_accuracy)
display('Training overall accuracy', train_accuracy.accuracy())

# Get a confusion matrix and overall accuracy for the validation sample.
validation_sample = validation_sample.classify(trained_classifier)
validation_accuracy = validation_sample.errorMatrix(label, 'classification')
display('Validation error matrix', validation_accuracy)
display('Validation accuracy', validation_accuracy.accuracy())
"""

# Classify the reflectance image from the trained classifier.
classified_image = standardized.classify(trained_classifier)

In [ ]:
Map = geemap.Map()
Map

In [ ]:
Map.addLayer(classified_image)

In [10]:
task_pb = ee.batch.Export.image.toDrive(
    image = classified_image,
    description = 'nm_lek_area_classification',
    folder = '!imagery', #!imagery
    region = feature_geom,
    scale = 1,
    crs = 'EPSG:26913', #6350 for albers and 26913 for nad83 utm zone 13 n
    maxPixels = 70000000000
)
task_pb.start()